# Weighted Hybrid Recommender for Yelp Restaurants

This notebook implements and evaluates four restaurant recommendation approaches:
- Model-based Collaborative Filtering (CF): learns user preferences from rating patterns
- Content-Based Filtering (CBF): matches users to restaurants based on category preferences
- Popularity baseline: recommends best-rated restaurants for cold-start users
- Weighted hybrid: blends CF and CBF via `hybrid_score = alpha * CF + (1 - alpha) * CBF`

Data is scoped to one city (default `Philadelphia`) for computational tractability.

**Outputs:**
- End-to-end recommendation workflow: data loading → model training → evaluation → interpretability

**Functionality:**
- Compares recommendation quality across four approaches using Top-K ranking metrics
- Provides per-user scores and confidence breakdowns for model transparency
- Includes mechanisms to avoid recommending previously-visited restaurants


## 1) Setup and experiment configuration

This section imports dependencies and defines the global configuration for data scope,
model behavior, and reproducibility.

**Outputs:**
- Printed paths and selected city (`Data dir`, `Target city`).
- The effective experiment settings (`K`, thresholds, sampling fraction).

**What it tells us:**
- Confirms run context before expensive processing starts.
- Makes the experiment reproducible and easy to compare across runs.

In [78]:
from __future__ import annotations

import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# Config
# -------------------------
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "original_data" / "yelp_json"
BUSINESS_PATH = DATA_DIR / "yelp_academic_dataset_business.json"
REVIEW_PATH = DATA_DIR / "yelp_academic_dataset_review.json"

# City for which recommendation is generated.
TARGET_CITY = "Philadelphia"
TARGET_STATE = "PA"
# K controls how many restaurants are returned/evaluated.
K = 10
# MIN_* thresholds remove very sparse users/items before modeling.
MIN_USER_INTERACTIONS = 3
MIN_BUSINESS_INTERACTIONS = 5
# User routing threshold: popularity fallback only for true cold-start users.
# All other users use weighted CF+CBF scoring.
ROUTE_POP_MAX_REVIEWS = 0
# Global blend weight for CF in hybrid scoring.
# hybrid_score = HYBRID_ALPHA * CF_norm + (1 - HYBRID_ALPHA) * CBF_norm
HYBRID_ALPHA = 0.9
# LIKE_STARS is the cutoff for "positive" interactions.
LIKE_STARS = 4.0
# Do not recommend businesses below this Yelp business rating.
# Keep equal to LIKE_STARS by default so relevance and serving quality targets align.
MIN_RECOMMEND_BUSINESS_STARS = LIKE_STARS
SEED = 42
# SAMPLE_USER_FRAC keeps experiments lightweight during iteration. Set to 1.0 for full data.
SAMPLE_USER_FRAC = 0.20  # 20% of users for faster experiments



np.random.seed(SEED)

print(f"Data dir: {DATA_DIR}")
print(f"Target city: {TARGET_CITY}")
print(f"Target state: {TARGET_STATE}")
print(
    "Routing bands (train reviews): "
    f"pop<= {ROUTE_POP_MAX_REVIEWS}, hybrid> {ROUTE_POP_MAX_REVIEWS}"
)
print(f"Hybrid alpha (CF weight): {HYBRID_ALPHA}")
print(f"Min business stars to recommend: {MIN_RECOMMEND_BUSINESS_STARS}")


Data dir: C:\Users\kello\PycharmProjects\ISA-Team-8\original_data\yelp_json
Target city: Philadelphia
Target state: PA
Routing bands (train reviews): pop<= 0, hybrid> 0
Hybrid alpha (CF weight): 0.9
Min business stars to recommend: 4.0


## 2) Data loading and preprocessing helpers

This section defines helper functions for streaming Yelp JSON files, parsing
categories, and applying interaction denoising/sampling.

**Outputs:**
- No direct table output; this cell defines reusable helper functions.

**What it tells us:**
- Documents how raw JSON is transformed into model-ready tables.
- Establishes filtering rules used later in evaluation and serving.

In [79]:
def iter_json_lines(path: Path):
    """Yield one JSON object per line from a Yelp JSONL file."""
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)


def parse_categories(raw: str | None) -> list[str]:
    """Split category text into a deduplicated, lowercase list."""
    if not raw:
        return []
    cats = [c.strip().lower() for c in raw.split(",") if c and c.strip()]
    return sorted(set(cats))


def load_city_restaurants(business_path: Path, city: str, state: str) -> pd.DataFrame:
    """Load only restaurant businesses from the selected city in a selected state."""
    records = []
    for obj in iter_json_lines(business_path):
        if str(obj.get("city", "")).strip() != city or str(obj.get("state", "")).strip() != state:
            continue

        cats = parse_categories(obj.get("categories"))
        if not cats:
            continue
        if not any("restaurant" in c for c in cats):
            continue

        records.append({
            "business_id": obj.get("business_id"),
            "name": obj.get("name"),
            "city": obj.get("city"),
            "state": obj.get("state"),
            "stars": float(obj.get("stars", 0.0)),
            "review_count": int(obj.get("review_count", 0)),
            "is_open": int(obj.get("is_open", 1)),
            "category_list": cats,
        })

    return pd.DataFrame(records)


def load_reviews_for_businesses(review_path: Path, business_ids: set[str]) -> pd.DataFrame:
    """Load reviews only for businesses that passed city/category filtering."""
    rows = []
    for obj in iter_json_lines(review_path):
        bid = obj.get("business_id")
        if bid not in business_ids:
            continue
        rows.append({
            "review_id": obj.get("review_id"),
            "user_id": obj.get("user_id"),
            "business_id": bid,
            "stars": float(obj.get("stars", 0.0)),
            "date": obj.get("date"),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df = df.dropna(subset=["date"])
    return df


def sample_users_by_fraction(review_df: pd.DataFrame, frac: float, seed: int) -> pd.DataFrame:
    """Randomly sample users and keep all of their reviews for faster runs."""
    if frac >= 1.0:
        return review_df
    users = review_df["user_id"].drop_duplicates().to_numpy()
    if users.size == 0:
        return review_df

    target_n = max(1, int(len(users) * frac))
    rng = np.random.default_rng(seed)
    sampled_users = set(rng.choice(users, size=target_n, replace=False).tolist())
    return review_df[review_df["user_id"].isin(sampled_users)].reset_index(drop=True)


def apply_interaction_filters(review_df: pd.DataFrame, min_u: int, min_b: int) -> pd.DataFrame:
    """Iteratively remove users/items below interaction thresholds."""
    df = review_df.copy()
    # Two passes are usually enough for this denoising step.
    for _ in range(2):
        user_ok = df["user_id"].value_counts()
        user_ok = set(user_ok[user_ok >= min_u].index)
        df = df[df["user_id"].isin(user_ok)]

        biz_ok = df["business_id"].value_counts()
        biz_ok = set(biz_ok[biz_ok >= min_b].index)
        df = df[df["business_id"].isin(biz_ok)]

    return df.reset_index(drop=True)


## 3) Build filtered dataset and temporal split

This section loads city-scoped restaurant data, samples users for faster iteration,
and creates train/validation/test splits by time.

**Outputs:**
- Printed counts for businesses, reviews, users, and train/val/test sizes.
- Sampling fraction used for this run.

**What it tells us:**
- Shows actual problem size and expected runtime cost.
- Verifies temporal split is populated correctly before modeling.

In [80]:
def temporal_split_leave_last_two(interactions: pd.DataFrame):
    """Per-user temporal split: train + optional val + test from latest records."""
    train_parts, val_parts, test_parts = [], [], []
    cohorts = {}

    for _, g in interactions.sort_values("date").groupby("user_id", sort=False):
        # Sort by date, then assign to cohorts based on interaction count for this user.
        n = len(g)
        if n >= 3:
            # Warm users: enough history for train/val/test split.
            train_parts.append(g.iloc[:-2])
            val_parts.append(g.iloc[-2:-1])
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "warm"
        elif n == 2:
            # Sparse users: only train (first) and test (last).
            train_parts.append(g.iloc[:1])
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "sparse"
        else:
            # Cold users: no train history, only test (for evaluation of popularity fallback).
            test_parts.append(g.iloc[-1:])
            cohorts[g.iloc[0]["user_id"]] = "cold"

    def _cat(parts):
        return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=interactions.columns)

    return _cat(train_parts), _cat(val_parts), _cat(test_parts), cohorts


business_df = load_city_restaurants(BUSINESS_PATH, TARGET_CITY, TARGET_STATE)
business_ids = set(business_df["business_id"].tolist())
review_df = load_reviews_for_businesses(REVIEW_PATH, business_ids)
# First denoising pass removes extreme sparsity before optional sampling.
review_df = apply_interaction_filters(review_df, MIN_USER_INTERACTIONS, MIN_BUSINESS_INTERACTIONS)
review_df = sample_users_by_fraction(review_df, SAMPLE_USER_FRAC, SEED)

# Re-apply denoising after sampling to keep matrix quality stable.
review_df = apply_interaction_filters(review_df, min_u=2, min_b=3)

# Keep only businesses that survived interaction denoising.
active_business_ids = set(review_df["business_id"].unique())
business_df = business_df[business_df["business_id"].isin(active_business_ids)].copy()

train_df, val_df, test_df, user_cohort = temporal_split_leave_last_two(review_df)

print("Businesses in scope:", len(business_df))
print("Reviews after filtering:", len(review_df))
print("Users after filtering:", review_df['user_id'].nunique())
print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
print("Sampling fraction:", SAMPLE_USER_FRAC)


Businesses in scope: 4062
Reviews after filtering: 96669
Users after filtering: 10091
Train/Val/Test: 76644 9934 10091
Sampling fraction: 0.2


## 4) Collaborative filtering model (CF)

This section builds a sparse user-item matrix and applies `TruncatedSVD` as a
model-based collaborative filtering method that learns latent user/item factors
for ranking.

**Outputs:**
- Printed latent factor shapes (`user_factors`, `item_factors`).

**What it tells us:**
- Confirms the model-based CF pipeline trained successfully.
- Helps detect under/over-sized latent space early.

In [81]:
all_users = sorted(review_df["user_id"].unique().tolist())
all_items = sorted(business_df["business_id"].unique().tolist())

# Build lookup mappings between user/item IDs and matrix indices for fast access.
u2i = {u: i for i, u in enumerate(all_users)}
b2i = {b: i for i, b in enumerate(all_items)}
i2u = {i: u for u, i in u2i.items()}
i2b = {i: b for b, i in b2i.items()}

def build_interaction_matrix(df: pd.DataFrame, user_map: dict[str, int], item_map: dict[str, int]) -> csr_matrix:
    """Convert interactions into a sparse user-item matrix for CF.

    Most user-item pairs have no interaction (user hasn't rated that restaurant).
    Dense matrix would be 99%+ zeros and waste memory. Sparse format (CSR) stores only nonzero values
    and their indices, reducing memory for typical data. SVD is efficient on sparse matrices.
    """
    if df.empty:
        return csr_matrix((len(user_map), len(item_map)), dtype=np.float32)

    # Map user/item IDs to matrix indices and extract star ratings as values.
    rows = df["user_id"].map(user_map).to_numpy()
    cols = df["business_id"].map(item_map).to_numpy()
    vals = df["stars"].astype(np.float32).to_numpy()
    # Build sparse matrix: (i,j) = row user, col item, value = rating.
    return csr_matrix((vals, (rows, cols)), shape=(len(user_map), len(item_map)), dtype=np.float32)


R_train = build_interaction_matrix(train_df, u2i, b2i)

# Keep latent dimension bounded so SVD remains stable on sampled data.
# Use smaller of 64 or (smallest matrix dimension - 1) to avoid numerical issues.
n_components = int(min(64, max(2, min(R_train.shape) - 1)))
# Truncated SVD decomposes R_train into user_factors @ item_factors.T for ranking.
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
user_factors = svd.fit_transform(R_train)
# Transpose for later: each column is one item's latent vector.
item_factors = svd.components_.T

print("CF factors:", user_factors.shape, item_factors.shape)


CF factors: (10091, 64) (4062, 64)


## 5) Content-based model (CBF) from categories

This section builds item category vectors and user preference profiles from training
interactions.

This content-based model uses restaurant categories only, which is lightweight
but limited compared with richer metadata (e.g., text, price, or other attributes).

**Outputs:**
- Printed item-category matrix shape and vocabulary size.
- In-memory user profile vectors (`user_profiles`).

**What it tells us:**
- Confirms category feature space quality and scale.
- Indicates whether category-only CBF has enough signal to personalize.

In [82]:
biz_cat = business_df[["business_id", "category_list"]].copy()
# Deduplicate by business_id only because category_list is a list (unhashable for full dedup).
biz_cat = biz_cat.drop_duplicates(subset=["business_id"], keep="first").set_index("business_id")
ordered_categories = [biz_cat["category_list"].get(b, []) for b in all_items]

# Multi-label binarizer converts category lists to one-hot vectors for each restaurant.
mlb = MultiLabelBinarizer()
X_item_cat = mlb.fit_transform(ordered_categories).astype(np.float32)
# Precompute L2 norms of category vectors for cosine similarity normalization later.
item_cat_norm = np.linalg.norm(X_item_cat, axis=1) + 1e-12  # Small epsilon to avoid division by zero.

train_user_groups = train_df.groupby("user_id")
user_profiles = {}
# Build one category-profile per user from training interactions.
# Positive interactions (>= LIKE_STARS) preferred; fallback uses all if none liked.
for uid in all_users:
    if uid not in train_user_groups.groups:
        # User has no training interactions; mark profile as None.
        user_profiles[uid] = None
        continue

    g = train_user_groups.get_group(uid)
    # Extract liked restaurants (above LIKE_STARS threshold).
    liked = g[g["stars"] >= LIKE_STARS]
    # Use liked restaurants if available, else use all interactions for signal.
    src = liked if not liked.empty else g

    # Map business IDs to item indices in the category matrix.
    idxs = [b2i[b] for b in src["business_id"].tolist() if b in b2i]
    if not idxs:
        # No valid restaurants; profile remains None.
        user_profiles[uid] = None
        continue

    # Use star ratings as weights: higher-rated restaurants influence profile more.
    w = src["stars"].astype(np.float32).to_numpy()
    if len(w) != len(idxs):
        # Fallback: uniform weights if mismatch (shouldn't happen with clean data).
        w = np.ones(len(idxs), dtype=np.float32)

    # Extract category vectors for this user's liked restaurants.
    mat = X_item_cat[idxs]
    # Weighted average of category vectors; higher-rated restaurants get more influence.
    profile = np.average(mat, axis=0, weights=w)
    # L2 normalize the profile for cosine similarity comparison later.
    norm = np.linalg.norm(profile) + 1e-12  # Epsilon prevents division by zero.
    user_profiles[uid] = profile / norm

print("CBF item matrix:", X_item_cat.shape, "vocab size:", len(mlb.classes_))


CBF item matrix: (4062, 335) vocab size: 335


## 6) Recommendation functions

This section defines ranking functions for CF, CBF, weighted hybrid scoring, and
popularity fallback.
For offline evaluation we exclude only training interactions.
For user-facing recommendations we exclude all known visited restaurants.

**Outputs:**
- No direct table output; this cell defines recommendation behavior.

**What it tells us:**
- Separates evaluation-time logic from serving-time non-repeat policy.
- Ensures deployed recommendations do not include previously visited places.

In [83]:
train_seen = defaultdict(set)
for r in train_df.itertuples(index=False):
    train_seen[r.user_id].add(r.business_id)

# Full-history visited set used for serving-time non-repeat recommendations.
all_seen = defaultdict(set)
for r in review_df.itertuples(index=False):
    all_seen[r.user_id].add(r.business_id)

# Precompute which items are eligible for recommendation by business rating.
business_star_map = business_df.set_index("business_id")["stars"].to_dict()
eligible_items = {b for b in all_items if business_star_map.get(b, 0.0) >= MIN_RECOMMEND_BUSINESS_STARS}
if not eligible_items:
    # Safety fallback: if threshold is too strict, avoid empty recommendations.
    eligible_items = set(all_items)

ineligible_item_idx = [b2i[b] for b in all_items if b not in eligible_items]

# Average train rating per business, reused by popularity fallback and confidence reporting.
popularity_score_map = train_df.groupby("business_id")["stars"].mean().to_dict()
pop_rank = train_df.groupby("business_id")["stars"].mean().sort_values(ascending=False).index.tolist()
popularity_rank = [b for b in pop_rank if b in eligible_items]

# Cold fill guarantees enough candidates even if some eligible items have no train interactions.
if len(popularity_rank) < K:
    eligible_by_business_stars = (
        business_df[business_df["business_id"].isin(eligible_items)]
        .sort_values("stars", ascending=False)["business_id"]
        .tolist()
    )
    seen_pop = set(popularity_rank)
    popularity_rank.extend([b for b in eligible_by_business_stars if b not in seen_pop])

def top_k_from_scores(scores: np.ndarray, seen: set[str], k: int) -> list[str]:
    """Return top-k item ids after masking already-seen businesses.

    Quality gate ensures only restaurants meeting MIN_RECOMMEND_BUSINESS_STARS
    are returned. Seen-items masking prevents recommending places user already visited
    (evaluation: training set only; serving: entire history). Masking is done by setting
    scores to -inf so argpartition naturally excludes them.

    Argpartition is faster than full sort when k << n_items. Argpartition finds K largest
    in O(n) expected time; full sort is O(n log n). For large catalogs, this saves significant compute.
    """
    if scores.size == 0:
        return []

    work = scores.copy()
    if ineligible_item_idx:
        # Apply quality gate: mask ineligible restaurants (below MIN_RECOMMEND_BUSINESS_STARS).
        work[ineligible_item_idx] = -np.inf

    if seen:
        # Mask already-visited restaurants so they do not appear in top-K results.
        seen_idx = [b2i[b] for b in seen if b in b2i]
        work[seen_idx] = -np.inf

    k = int(min(k, len(work)))
    if k <= 0:
        return []

    # Use argpartition for efficient top-K extraction (faster than full sort).
    cand = np.argpartition(work, -k)[-k:]
    # Sort the K candidates by score descending to return ranked list.
    cand = cand[np.argsort(work[cand])[::-1]]
    # Convert indices back to business IDs, filtering out masked items.
    return [i2b[int(i)] for i in cand if np.isfinite(work[int(i)])]


def _safe_minmax(values: np.ndarray) -> np.ndarray:
    """Normalize values to [0,1] range for fair score blending.

    CF and CBF use different scoring scales (latent factors vs cosine similarity).
    Blending raw scores would favor whichever method has larger magnitude.
    Min-max scaling to [0,1] ensures both contribute equally to the final hybrid score
    regardless of their original scale.

    Handles edge case where all values are constant by returning 0.5 (neutral score).
    """
    if values.size == 0:
        return values
    # Min-max normalization: (x - min) / (max - min) rescales to [0, 1].
    vmin = float(np.min(values))
    vmax = float(np.max(values))
    # If all values are constant, return 0.5 to avoid division by zero.
    if vmax - vmin < 1e-12:
        return np.full(values.shape, 0.5, dtype=np.float32)
    return ((values - vmin) / (vmax - vmin)).astype(np.float32)


def recommend_cf(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """CF ranking from latent factors with popularity fallback.

    Latent factor dot product measures user-item alignment in learned preference space.
    Users with similar preferences are close in this space, so their preferred items will
    score high for each other. Returns popularity fallback for new users without latent factors.
    """
    if uid not in u2i:
        return popularity_rank[:k]

    uidx = u2i[uid]
    scores = user_factors[uidx] @ item_factors.T
    seen = train_seen.get(uid, set()) if seen_override is None else seen_override
    recs = top_k_from_scores(scores, seen, k)
    return recs if recs else popularity_rank[:k]


def recommend_cbf(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """CBF ranking from category-profile similarity with fallback.

    User profile is learned as weighted average of categories they liked.
    Items are vectors in category space. Cosine similarity measures how well each item
    matches the user's category preferences. High similarity = item categories align with
    restaurants this user historically enjoyed.
    """
    profile = user_profiles.get(uid)
    if profile is None:
        return popularity_rank[:k]

    scores = (X_item_cat @ profile) / item_cat_norm
    seen = train_seen.get(uid, set()) if seen_override is None else seen_override
    recs = top_k_from_scores(scores, seen, k)
    return recs if recs else popularity_rank[:k]


def cf_score_vector(uid: str) -> np.ndarray:
    """Return raw CF score vector for a user (or popularity proxy when unavailable)."""
    if uid not in u2i:
        return popularity_scores.copy()
    return (user_factors[u2i[uid]] @ item_factors.T).astype(np.float32)


def cbf_score_vector(uid: str) -> np.ndarray:
    """Return raw CBF score vector for a user (or popularity proxy when unavailable)."""
    profile = user_profiles.get(uid)
    if profile is None:
        return popularity_scores.copy()
    return ((X_item_cat @ profile) / item_cat_norm).astype(np.float32)


def hybrid_score_vector(uid: str, alpha: float = HYBRID_ALPHA) -> np.ndarray:
    """Blend normalized CF/CBF score vectors into one hybrid score vector.

    CF captures user-user similarities and collaborative patterns (what similar users liked).
    CBF captures user-item feature alignment (what categories this user prefers).
    Blending combines both signals to capture both "people like you enjoyed X" (CF)
    and "this matches your taste profile" (CBF).

    Normalization is critical: before blending, both vectors are scaled to [0,1] so that
    neither method dominates based on raw magnitude. Alpha controls the trade-off:
    - Higher alpha (0.65): trust CF more; good when user has rich history.
    - Lower alpha (0.35): equal weight; good when signals are uncertain.
    """
    cf_norm = _safe_minmax(cf_score_vector(uid))
    cbf_norm = _safe_minmax(cbf_score_vector(uid))
    return (alpha * cf_norm + (1.0 - alpha) * cbf_norm).astype(np.float32)


def recommend_hybrid(uid: str, k: int = K, seen_override: set[str] | None = None) -> list[str]:
    """Recommend with popularity for true cold start, otherwise weighted CF+CBF.

    Cold-start users (0 reviews) have no history for CF or CBF to learn from.
    Popularity is the only signal available: recommend best-rated restaurants everyone likes.
    Users with any training history benefit from hybrid blending of CF+CBF signals.
    Fallback to popularity_rank ensures we always return K recommendations even if scoring fails.
    """
    seen = train_seen.get(uid, set()) if seen_override is None else seen_override

    # For true cold-start users (0 train reviews), use popularity-only fallback.
    if train_counts.get(uid, 0) <= ROUTE_POP_MAX_REVIEWS:
        recs = top_k_from_scores(popularity_scores, seen, k)
        return recs if recs else popularity_rank[:k]

    # For all other users, use weighted blend of normalized CF and CBF scores.
    scores = hybrid_score_vector(uid, alpha=HYBRID_ALPHA)
    recs = top_k_from_scores(scores, seen, k)
    return recs if recs else popularity_rank[:k]


def recommend_hybrid_serving(uid: str, k: int = K) -> list[str]:
    """Serving wrapper: never return already visited restaurants."""
    # Serving policy: never recommend restaurants already visited by this user.
    return recommend_hybrid(uid, k, seen_override=all_seen.get(uid, set()))


## 7) Metrics and hybrid logic

This section defines Top-K ranking metrics and weighted hybrid behavior.

Hybrid behavior here is:
- `popularity` only for true cold start (`n_train <= ROUTE_POP_MAX_REVIEWS`),
- weighted blend for all others: `alpha * CF + (1 - alpha) * CBF`.

**Outputs:**
- No direct DataFrame; defines metric functions and hybrid blending logic.
- Creates `val_rel`, `test_rel`, and `train_counts`.

**What it tells us:**
- Encodes how Top-K recommendation quality is measured.
- Encodes when popularity fallback is used and how CF/CBF are blended.

In [84]:
def dcg_at_k(binary_rel: list[int], k: int) -> float:
    """Discounted cumulative gain for a binary relevance list.

    Captures the gain from relevant items AND penalizes finding them later.
    Logarithmic discounting means rank 1 is worth 1.0, rank 2 is ~0.63, rank 10 is ~0.29.
    This reflects user behavior: most people check top results, few scroll to bottom.
    DCG naturally prefers systems that rank good items early.
    """
    vals = np.array(binary_rel[:k], dtype=np.float32)
    if vals.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, vals.size + 2))
    return float(np.sum(vals * discounts))


def ndcg_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Normalized DCG at K: ranking quality that rewards putting relevant items near the top.

    NDCG penalizes bad ranking order. A system that puts the good item at rank 10
    scores lower than one that puts it at rank 1, even if both hit the user's preference.
    DCG (Discounted Cumulative Gain) uses log discounting: earlier positions matter more.
    NDCG normalizes by ideal ranking to give a [0,1] score for fair comparison across users.
    """
    if not rel_set:
        return 0.0
    gains = [1 if r in rel_set else 0 for r in recs[:k]]
    dcg = dcg_at_k(gains, k)
    ideal = [1] * min(len(rel_set), k)
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else dcg / idcg


def precision_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Precision at K: fraction of top-K recommendations that are relevant.

    answers "how many of the K items I showed were actually good?"
    High precision = user wastes less time on bad recommendations.
    Important for user satisfaction in real systems.
    """
    if k <= 0:
        return 0.0
    hit = sum(1 for r in recs[:k] if r in rel_set)
    return float(hit / k)


def recall_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Recall at K: fraction of user's relevant items recovered in top-K.

    answers "how many of the items this user would like did I find?"
    High recall = successfully discovered a large portion of user's preferences.
    Complements precision: high precision + high recall = both accurate and comprehensive.
    """
    if not rel_set:
        return 0.0
    hit = sum(1 for r in recs[:k] if r in rel_set)
    return float(hit / len(rel_set))


def hit_rate_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Hit Rate at K: whether at least one relevant item appears in top-K.

    simplest measure of "did the system find anything useful for this user?"
    Binary outcome (0 or 1): useful for calculating what % of users get at least one good rec.
    A baseline retrieval metric: high hit rate is necessary but not sufficient (need precision too).
    """
    return 1.0 if any(r in rel_set for r in recs[:k]) else 0.0


def ap_at_k(recs: list[str], rel_set: set[str], k: int) -> float:
    """Average Precision at K: precision-at-position averaged across hit positions.

    Rewards both finding relevant items AND ranking them early.
    Combines benefits of precision and ranking order: if you find 3 good items,
    finding them at positions 1,3,5 (MAP~0.8) is better than 5,8,10 (MAP~0.35).
    """
    if not rel_set:
        return 0.0
    hit_count = 0
    score = 0.0
    for i, r in enumerate(recs[:k], start=1):
        if r in rel_set:
            hit_count += 1
            score += hit_count / i
    denom = min(len(rel_set), k)
    return score / denom if denom else 0.0


def build_relevance_by_user(df: pd.DataFrame, like_stars: float = LIKE_STARS) -> dict[str, set[str]]:
    """Build user->relevant-business set using a star cutoff.

    Why binary relevance: Top-K recommendation evaluation needs a ground truth.
    Restaurants rated >= LIKE_STARS are marked as relevant (positive preference signal).
    This simplifies metrics: a recommendation is either "hit" (relevant) or "miss" (not relevant).
    """
    if df.empty:
        return {}
    rel = defaultdict(set)
    for r in df.itertuples(index=False):
        if r.stars >= like_stars:
            rel[r.user_id].add(r.business_id)
    return dict(rel)


val_rel = build_relevance_by_user(val_df)
test_rel = build_relevance_by_user(test_df)
train_counts = train_df["user_id"].value_counts().to_dict()

# Precompute popularity score vector on the full item index for cold-start fallback.
popularity_scores = np.array([popularity_score_map.get(b, 0.0) for b in all_items], dtype=np.float32)


# `recommend_hybrid` is defined in section 6 with optional seen overrides.


## 8) Overall and cohort evaluation

This section compares Popularity, CF, CBF, and the Weighted Hybrid on overall
metrics and on warm/sparse/cold cohorts.

This notebook evaluates a Top-K recommendation task, not a rating-prediction task.
Therefore, the main metrics are ranking/retrieval metrics:
`Precision@10`, `Recall@10`, `HitRate@10`, `NDCG@10`, `MAP@10`, and `Coverage@10`.

**How to read the metrics (higher is better):**
- `Precision@10`: fraction of top-10 recommendations that are relevant.
- `Recall@10`: fraction of a user's relevant items recovered in top-10.
- `HitRate@10`: percent of users with at least one relevant item in top-10.
- `NDCG@10`: ranking quality that rewards putting relevant items near the top.
- `MAP@10`: average precision across top-10 ranking positions.
- `Coverage@10`: share of catalog that appears in recommendations (diversity/novelty proxy).

**What cohorts mean:**
- `warm`: users with enough history (>=3 interactions in this split).
- `sparse`: users with very limited history (exactly 2 interactions here).
- `cold`: users with no train history (not enough data for personalized training).

**About low-looking values:**
On large, sparse recommendation tasks, absolute values can look small.
The key signal is relative comparison between models and cohorts, not raw magnitude alone.

**Outputs:**
- `overall_df`: model comparison on all evaluable users.
- `cohort_df`: model comparison split by `warm`, `sparse`, `cold` users.
- `route_debug_df`: per-user hybrid audit (`train_review_count`, `mode`, `alpha_used`, `cohort`).
- `cohort_user_counts` and optional low-support warning table.

**What it tells us:**
- Identifies the best-performing model overall and per cohort.
- Tests whether weighted blending improves over single-model baselines.
- Shows whether results are reliable (or unstable) based on cohort support.

In [85]:
def evaluate_model(model_name: str, rec_fn, rel_by_user: dict[str, set[str]], k: int = K) -> dict:
    """Evaluate one recommender function on a relevance dictionary."""
    users = [u for u in rel_by_user.keys()]
    if not users:
        return {"model": model_name, "users": 0}

    # Initialize lists to accumulate metrics across all users.
    p_list, r_list, h_list, n_list, ap_list = [], [], [], [], []
    recommended_catalog = set()

    for uid in users:
        # Get recommendations and compute metrics for this user.
        recs = rec_fn(uid, k)
        rel = rel_by_user.get(uid, set())

        # Accumulate individual metrics for later averaging.
        p_list.append(precision_at_k(recs, rel, k))
        r_list.append(recall_at_k(recs, rel, k))
        h_list.append(hit_rate_at_k(recs, rel, k))
        n_list.append(ndcg_at_k(recs, rel, k))
        ap_list.append(ap_at_k(recs, rel, k))
        # Track all unique items recommended across all users for coverage metric.
        recommended_catalog.update(recs[:k])

    # Return model name and mean metrics across all users.
    return {
        "model": model_name,
        "users": len(users),
        f"Precision@{k}": float(np.mean(p_list)),
        f"Recall@{k}": float(np.mean(r_list)),
        f"HitRate@{k}": float(np.mean(h_list)),
        f"NDCG@{k}": float(np.mean(n_list)),
        f"MAP@{k}": float(np.mean(ap_list)),
        # Coverage: fraction of catalog items that appear in any top-K recommendation.
        f"Coverage@{k}": float(len(recommended_catalog) / max(1, len(all_items))),
    }


def recommend_pop(uid: str, k: int = K) -> list[str]:
    """Popularity baseline for comparison and fallback."""
    return popularity_rank[:k]


results = [
    # Overall comparison: every model evaluated on the same test relevance set.
    evaluate_model("Popularity", recommend_pop, test_rel, K),
    evaluate_model("CF", recommend_cf, test_rel, K),
    evaluate_model("CBF", recommend_cbf, test_rel, K),
    evaluate_model("Hybrid", recommend_hybrid, test_rel, K),
]

overall_df = pd.DataFrame(results).sort_values(by=f"NDCG@{K}", ascending=False).reset_index(drop=True)
display(overall_df)

# Hybrid-audit table: verifies cold-start fallback vs weighted blend usage.
route_debug_rows = []
for uid in test_rel.keys():
    n_train = int(train_counts.get(uid, 0))
    mode = "pop" if n_train <= ROUTE_POP_MAX_REVIEWS else "weighted_hybrid"
    alpha_used = 0.0 if mode == "pop" else HYBRID_ALPHA

    route_debug_rows.append({
        "user_id": uid,
        "train_review_count": n_train,
        "cohort": user_cohort.get(uid, "unknown"),
        "mode": mode,
        "alpha_used": alpha_used,
        "cf_weight": alpha_used,
        "cbf_weight": 1.0 - alpha_used,
    })

route_debug_df = pd.DataFrame(route_debug_rows)
display(route_debug_df.head(10))
display(
    route_debug_df.groupby("mode", as_index=False)
    .size()
    .rename(columns={"size": "user_count"})
    .sort_values("user_count", ascending=False)
)

# Per-cohort evaluation: test each model separately on warm, sparse, cold users.
# This shows whether model performance varies by user history availability.
def filter_rel_by_users(rel_by_user: dict[str, set[str]], users: set[str]) -> dict[str, set[str]]:
    """Restrict relevance mapping to a target user subset."""
    # Keep only relevance entries for users in the target cohort.
    return {u: rel_by_user[u] for u in rel_by_user if u in users}

# Partition test users into cohorts based on their training history size.
warm_users = {u for u, c in user_cohort.items() if c == "warm"}
sparse_users = {u for u, c in user_cohort.items() if c == "sparse"}
cold_users = {u for u, c in user_cohort.items() if c == "cold"}

cohort_rows = []
# For each cohort, evaluate all models on that cohort's test set.
for cohort_name, cohort_users in [("warm", warm_users), ("sparse", sparse_users), ("cold", cold_users)]:
    rel_subset = filter_rel_by_users(test_rel, cohort_users)
    # Evaluate all four models on this cohort's subset of test data.
    cohort_rows.extend([
        {"cohort": cohort_name, **evaluate_model("Popularity", recommend_pop, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("CF", recommend_cf, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("CBF", recommend_cbf, rel_subset, K)},
        {"cohort": cohort_name, **evaluate_model("Hybrid", recommend_hybrid, rel_subset, K)},
    ])

cohort_df = pd.DataFrame(cohort_rows)
cohort_df.sort_values(["cohort", f"NDCG@{K}"], ascending=[True, False]).reset_index(drop=True)

# Quick cohort support check: useful when NaNs appear from tiny/empty cohorts.
cohort_user_counts = pd.Series(user_cohort).value_counts().rename_axis("cohort").reset_index(name="user_count")
display(cohort_user_counts)

low_support = cohort_user_counts[cohort_user_counts["user_count"] < 30]
if not low_support.empty:
    print("Warning: some cohorts have low support (<30 users), metrics can be unstable.")
    display(low_support)


,model,users,Precision@10,Recall@10,HitRate@10,NDCG@10,MAP@10,Coverage@10
0,Hybrid,7284,0.003693,0.036930,0.036930,0.018278,0.012680,0.234367
1,CF,7284,0.003679,0.036793,0.036793,0.018115,0.012517,0.121861
2,CBF,7284,0.000659,0.006590,0.006590,0.003058,0.002018,0.477597
3,Popularity,7284,0.000124,0.001236,0.001236,0.000526,0.000312,0.002462


,user_id,train_review_count,cohort,mode,alpha_used,cf_weight,cbf_weight
0,H4JNrBAoyCk_ZMZWbAf8OA,13,warm,weighted_hybrid,0.9,0.9,0.1
1,PO-U11FmTDiqCEqtilFjVQ,11,warm,weighted_hybrid,0.9,0.9,0.1
2,GitE04dW0jYxH5LplUTDBA,3,warm,weighted_hybrid,0.9,0.9,0.1
3,DLrptiHwqa0JLYFJsp4vzg,2,warm,weighted_hybrid,0.9,0.9,0.1
4,gT5u8PAIaWQOYbf53t91KQ,1,warm,weighted_hybrid,0.9,0.9,0.1
5,qCNZXu0nA1m9_qQDSY9bfA,2,warm,weighted_hybrid,0.9,0.9,0.1
6,wZJEiUtMGlUhqHqFKXLp5g,1,warm,weighted_hybrid,0.9,0.9,0.1
7,hLym9La5ivWSqkDqiY4kFA,2,warm,weighted_hybrid,0.9,0.9,0.1
8,xfe25SzgOvx5BUlXVg4YWg,14,warm,weighted_hybrid,0.9,0.9,0.1
9,fJF2xokaIgQsmNLuav5etQ,10,warm,weighted_hybrid,0.9,0.9,0.1


,mode,user_count
0,weighted_hybrid,7284


,cohort,user_count
0,warm,9934
1,sparse,157


## 9) Basic recommendation preview

This section returns a simple recommendation table for a sample user.
These recommendations use serving-time exclusions (no already visited restaurants).

**Outputs:**
- `sample_users`: a few user ids from the test relevance set.
- A compact top-K recommendation table (`rank`, `mode`, `alpha_used`, business info).

**What it tells us:**
- Quick sanity check that end-to-end recommendation runs.
- Human-readable qualitative preview before deeper confidence analysis.

In [86]:
def recommendation_table(uid: str, k: int = 10) -> pd.DataFrame:
    """Simple serving-time recommendation table for a user."""
    n_train = train_counts.get(uid, 0)
    mode = "pop" if n_train <= ROUTE_POP_MAX_REVIEWS else "weighted_hybrid"
    alpha_used = 0.0 if mode == "pop" else HYBRID_ALPHA
    recs = recommend_hybrid_serving(uid, k)

    out = business_df[business_df["business_id"].isin(recs)][["business_id", "name", "city", "stars", "category_list"]].copy()
    out["rank"] = out["business_id"].map({b: i + 1 for i, b in enumerate(recs)})
    out["mode"] = mode
    out["alpha_used"] = alpha_used
    out = out.sort_values("rank")
    return out[["rank", "mode", "alpha_used", "business_id", "name", "city", "stars", "category_list"]]


sample_users = list(test_rel.keys())[:3]
sample_users


['H4JNrBAoyCk_ZMZWbAf8OA', 'PO-U11FmTDiqCEqtilFjVQ', 'GitE04dW0jYxH5LplUTDBA']

## 10) Confidence report for each recommendation

This section provides heuristic interpretability scores at two levels:
- **hybrid confidence**: how much trust to place in weighted blending vs cold-start fallback
- **item confidence**: relative score strength within the returned top-K list

These values are explanatory heuristics for inspection,
not calibrated statistical confidence estimates.

The confidence table also applies serving-time non-repeat filtering.

**Outputs:**
- `recommendation_confidence_table(...)` with per-item confidence columns:
  - `item_confidence`, `raw_score`, `model_confidence`, `model_reason`, `selection_reason`,
    `cf_component`, `cbf_component`, `already_visited`.
- A rendered confidence table for one sample user.

**What it tells us (most important):**
- How strongly the selected model appears supported for that user (`model_confidence`).
- Which recommended restaurants are stronger/weaker within the returned list (`item_confidence`).
- Why the model was selected and what scoring logic produced the final ranking.
- Whether serving constraints are respected (`already_visited` should be `False`).

In [87]:
def model_selection_confidence(uid: str) -> tuple[float, str]:
    """Heuristic interpretability score for the selected recommendation model."""
    n_train = train_counts.get(uid, 0)

    if n_train <= ROUTE_POP_MAX_REVIEWS:
        conf = min(0.70, 0.45 + 0.04 * n_train)
        return conf, (
            f"{n_train} train reviews (<= {ROUTE_POP_MAX_REVIEWS}): "
            "use popularity baseline (best-rated restaurants)."
        )

    conf = min(0.92, 0.62 + min(0.30, n_train / 100.0))
    return conf, (
        f"{n_train} train reviews (> {ROUTE_POP_MAX_REVIEWS}): "
        f"use weighted blend alpha={HYBRID_ALPHA:.2f} (CF) and {1.0 - HYBRID_ALPHA:.2f} (CBF)."
    )


def recommendation_confidence_table(uid: str, k: int = 10) -> pd.DataFrame:
    """Detailed serving-time table with item and model confidence fields."""
    n_train = train_counts.get(uid, 0)
    # Determine mode (pop fallback or weighted blend) based on user's training history.
    mode = "pop" if n_train <= ROUTE_POP_MAX_REVIEWS else "weighted_hybrid"
    # Alpha used in recommendation: 0 for pop, HYBRID_ALPHA for blend.
    alpha_used = 0.0 if mode == "pop" else HYBRID_ALPHA
    # Exclude all previously visited restaurants from serving recommendations.
    seen_for_serving = all_seen.get(uid, set())

    # Compute scores and component breakdowns based on the selected mode.
    if mode == "pop":
        # Cold-start: use pure popularity scores.
        scores = popularity_scores.copy()
        # No CF/CBF components in popularity mode.
        cf_component = np.zeros(len(all_items), dtype=np.float32)
        cbf_component = np.zeros(len(all_items), dtype=np.float32)
        model_reason = "Cold-start fallback: popularity scores from city average rating."
    else:
        # Weighted blend: compute and store both CF and CBF normalized components.
        cf_norm = _safe_minmax(cf_score_vector(uid))
        cbf_norm = _safe_minmax(cbf_score_vector(uid))
        # Scale components by alpha for transparency in recommendation breakdown.
        cf_component = HYBRID_ALPHA * cf_norm
        cbf_component = (1.0 - HYBRID_ALPHA) * cbf_norm
        # Final score is the sum of weighted components.
        scores = (cf_component + cbf_component).astype(np.float32)
        model_reason = "Weighted hybrid combines normalized CF and CBF score components."

    recs = top_k_from_scores(scores, seen_for_serving, k)
    # Normalize item confidence relative to the top-K scores (not absolute scale).
    rec_scores = np.array([float(scores[b2i[b]]) for b in recs], dtype=np.float32)
    # Item confidence: 0=weakest, 1=strongest within this top-K list.
    rec_conf = _safe_minmax(rec_scores)

    # Get model selection confidence for this user.
    model_conf, selection_reason = model_selection_confidence(uid)

    # Build result table: business info + scores + confidence + components + reasons.
    base = business_df[business_df["business_id"].isin(recs)][
        ["business_id", "name", "city", "stars", "category_list"]
    ].copy()
    base["rank"] = base["business_id"].map({b: i + 1 for i, b in enumerate(recs)})
    base["raw_score"] = base["business_id"].map({b: s for b, s in zip(recs, rec_scores)})
    base["item_confidence"] = base["business_id"].map({b: c for b, c in zip(recs, rec_conf)})
    # Store individual CF and CBF contributions for interpretability.
    base["cf_component"] = base["business_id"].map({b: float(cf_component[b2i[b]]) for b in recs})
    base["cbf_component"] = base["business_id"].map({b: float(cbf_component[b2i[b]]) for b in recs})
    base["mode"] = mode
    base["alpha_used"] = float(alpha_used)
    base["model_confidence"] = float(model_conf)
    base["model_reason"] = model_reason
    base["selection_reason"] = selection_reason
    # Flag already-visited restaurants (should always be False in serving).
    base["already_visited"] = base["business_id"].isin(seen_for_serving)

    base = base.sort_values("rank")
    return base[[
        "rank",
        "mode",
        "alpha_used",
        "model_confidence",
        "item_confidence",
        "raw_score",
        "cf_component",
        "cbf_component",
        "already_visited",
        "business_id",
        "name",
        "city",
        "stars",
        "category_list",
        "model_reason",
        "selection_reason",
    ]]


if sample_users:
    display(recommendation_confidence_table(sample_users[0], k=10))
else:
    print("No evaluable users found in test relevance set.")


,rank,mode,alpha_used,model_confidence,item_confidence,raw_score,cf_component,cbf_component,already_visited,business_id,name,city,stars,category_list,model_reason,selection_reason
2442,1,weighted_hybrid,0.9,0.75,1.000000,0.295282,0.255150,0.040132,False,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,4.0,"[american (new), breakfast & brunch, french, i...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
2135,2,weighted_hybrid,0.9,0.75,0.800920,0.284461,0.216328,0.068133,False,vpNJJNVgVmf2u1lfdFh81w,Matyson,Philadelphia,4.0,"[american (new), breakfast & brunch, restaurants]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
2788,3,weighted_hybrid,0.9,0.75,0.710389,0.279540,0.221851,0.057690,False,R17gwW6zn9ilslbdvKdgsg,Alma de Cuba,Philadelphia,4.0,"[bars, caribbean, cocktail bars, cuban, latin ...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
21,4,weighted_hybrid,0.9,0.75,0.612762,0.274234,0.200930,0.073304,False,7mpYTDb24SywNMRn3yeakQ,The Twisted Tail,Philadelphia,4.0,"[american (new), american (traditional), bars,...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
2044,5,weighted_hybrid,0.9,0.75,0.578992,0.272398,0.238444,0.033954,False,wbDRmtxaKRpBOjutvV6TEA,Barclay Prime,Philadelphia,4.5,"[restaurants, steakhouses]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
5378,6,weighted_hybrid,0.9,0.75,0.522530,0.269329,0.208141,0.061188,False,OdIBX09glfXNVSyd0RnIeg,Monk's Cafe,Philadelphia,4.0,"[bars, belgian, gastropubs, nightlife, pubs, r...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
1016,7,weighted_hybrid,0.9,0.75,0.276381,0.255949,0.200312,0.055638,False,6ewV-e7-39oqYUq3yZuIyw,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
2501,8,weighted_hybrid,0.9,0.75,0.154606,0.249330,0.212418,0.036912,False,Qw7tz-UkPrpXaVidWuab4Q,Philadelphia Museum of Art,Philadelphia,4.5,"[american (new), art galleries, art museums, a...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
3836,9,weighted_hybrid,0.9,0.75,0.124513,0.247694,0.175350,0.072344,False,i_FWONQD1ZBqrNE2b-M5Ug,Talula's Garden,Philadelphia,4.5,"[american (new), restaurants]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...
4014,10,weighted_hybrid,0.9,0.75,0.000000,0.240926,0.173386,0.067541,False,jHVotJhyPKOLuO7MVnmbtQ,Royal Tavern,Philadelphia,4.0,"[american (new), bars, cocktail bars, nightlif...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...


## 11) Compare recommendations to user positive history

This section helps qualitative interpretation by comparing recommended restaurants to places
the user rated positively in the past (using category overlap).

**Outputs:**
- `rec_compare_df`: recommendation table augmented with:
  - `liked_category_overlap`, `liked_category_overlap_ratio`.
- `liked_history_df`: user's positively rated restaurants sorted by stars.

**What it tells us (most important):**
- How aligned each recommendation is with the user's known likes.
- Whether the system is discovering related categories or drifting away from preferences.
- An explanation layer for review, complementary to offline ranking metrics.


In [88]:
# Build positive history from full filtered data so explanation tables have enough context.
positive_history = build_relevance_by_user(review_df, like_stars=LIKE_STARS)


def compare_recommendations_to_positive_history(uid: str, k: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare recommendations to user's liked-history via category overlap."""
    rec_df = recommendation_confidence_table(uid, k=k).copy()
    # Get all restaurants this user has liked historically (across full dataset).
    liked_ids = positive_history.get(uid, set())

    # Retrieve restaurant metadata for user's liked restaurants.
    liked_df = business_df[business_df["business_id"].isin(liked_ids)][
        ["business_id", "name", "city", "stars", "category_list"]
    ].copy()

    # Aggregate all categories from restaurants the user has liked.
    # This represents the user's overall category preferences.
    liked_categories = set()
    if not liked_df.empty:
        for cats in liked_df["category_list"]:
            liked_categories.update(cats)

    def overlap_count(cats: list[str]) -> int:
        """Count overlapping categories with the user's positively rated history."""
        # Intersection of this recommendation's categories with user's preferred categories.
        return len(set(cats).intersection(liked_categories))

    # Compute category overlap for each recommendation.
    rec_df["liked_category_overlap"] = rec_df["category_list"].apply(overlap_count)
    # Normalize overlap by the recommendation's own category count (0 if no categories).
    # Ratio interpretation: 1.0 = all its categories are preferred; 0 = none match.
    rec_df["liked_category_overlap_ratio"] = rec_df["liked_category_overlap"] / rec_df["category_list"].apply(lambda x: max(1, len(x)))

    # Sort liked history by rating to show user's top preferences first.
    liked_df = liked_df.sort_values("stars", ascending=False)
    return rec_df, liked_df


if sample_users:
    rec_compare_df, liked_history_df = compare_recommendations_to_positive_history(sample_users[0], k=10)
    display(rec_compare_df)
    display(liked_history_df.head(15))
else:
    print("No evaluable users found in test relevance set.")


,rank,mode,alpha_used,model_confidence,item_confidence,raw_score,cf_component,cbf_component,already_visited,business_id,name,city,stars,category_list,model_reason,selection_reason,liked_category_overlap,liked_category_overlap_ratio
2442,1,weighted_hybrid,0.9,0.75,1.000000,0.295282,0.255150,0.040132,False,fOhnSqmO4XY5vSI8whVKSA,Farmicia,Philadelphia,4.0,"[american (new), breakfast & brunch, french, i...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,3,0.500000
2135,2,weighted_hybrid,0.9,0.75,0.800920,0.284461,0.216328,0.068133,False,vpNJJNVgVmf2u1lfdFh81w,Matyson,Philadelphia,4.0,"[american (new), breakfast & brunch, restaurants]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,3,1.000000
2788,3,weighted_hybrid,0.9,0.75,0.710389,0.279540,0.221851,0.057690,False,R17gwW6zn9ilslbdvKdgsg,Alma de Cuba,Philadelphia,4.0,"[bars, caribbean, cocktail bars, cuban, latin ...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,6,0.857143
21,4,weighted_hybrid,0.9,0.75,0.612762,0.274234,0.200930,0.073304,False,7mpYTDb24SywNMRn3yeakQ,The Twisted Tail,Philadelphia,4.0,"[american (new), american (traditional), bars,...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,6,0.750000
2044,5,weighted_hybrid,0.9,0.75,0.578992,0.272398,0.238444,0.033954,False,wbDRmtxaKRpBOjutvV6TEA,Barclay Prime,Philadelphia,4.5,"[restaurants, steakhouses]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,1,0.500000
5378,6,weighted_hybrid,0.9,0.75,0.522530,0.269329,0.208141,0.061188,False,OdIBX09glfXNVSyd0RnIeg,Monk's Cafe,Philadelphia,4.0,"[bars, belgian, gastropubs, nightlife, pubs, r...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,4,0.666667
1016,7,weighted_hybrid,0.9,0.75,0.276381,0.255949,0.200312,0.055638,False,6ewV-e7-39oqYUq3yZuIyw,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,6,0.857143
2501,8,weighted_hybrid,0.9,0.75,0.154606,0.249330,0.212418,0.036912,False,Qw7tz-UkPrpXaVidWuab4Q,Philadelphia Museum of Art,Philadelphia,4.5,"[american (new), art galleries, art museums, a...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,4,0.400000
3836,9,weighted_hybrid,0.9,0.75,0.124513,0.247694,0.175350,0.072344,False,i_FWONQD1ZBqrNE2b-M5Ug,Talula's Garden,Philadelphia,4.5,"[american (new), restaurants]",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,2,1.000000
4014,10,weighted_hybrid,0.9,0.75,0.000000,0.240926,0.173386,0.067541,False,jHVotJhyPKOLuO7MVnmbtQ,Royal Tavern,Philadelphia,4.0,"[american (new), bars, cocktail bars, nightlif...",Weighted hybrid combines normalized CF and CBF...,13 train reviews (> 0): use weighted blend alp...,7,0.777778


,business_id,name,city,stars,category_list
3487,6_T2xzR74JqGCTPefAD8Tw,Morimoto,Philadelphia,4.5,"[american (new), american (traditional), asian..."
2231,ZKPrXH_GNW_AtZ31tP3NmA,White Dog Cafe,Philadelphia,4.0,"[american (new), american (traditional), bars,..."
3591,aXr74YWu67vUtWsbmICbag,Rose Tattoo Cafe,Philadelphia,4.0,"[american (new), american (traditional), bars,..."
3699,55gCXlWDDCdttR3yRss1xw,The Rittenhouse Hotel,Philadelphia,4.0,"[event planning & services, hotels, hotels & t..."
4136,K3RURR9lIEE4JjOaPt99zg,Sabrina's Café,Philadelphia,4.0,"[american (new), american (traditional), break..."
5033,nIAbuktMEzVjT4P9pG89rQ,Buddakan,Philadelphia,4.0,"[asian fusion, chinese, restaurants]"
5681,JHRlwxxKY0JJcU97rJ-Bug,Cuba Libre Restaurant & Rum Bar - Philadelphia,Philadelphia,3.5,"[bars, beer, breakfast & brunch, cuban, dance ..."
1969,Co3Ogqy6y2JgZdG0wBlrUQ,Ten Stone Bar & Restaurant,Philadelphia,3.0,"[american (new), asian fusion, bars, nightlife..."
5253,bAqDJlYqB9j3tMVjhp29EQ,Mezze,Philadelphia,3.0,"[food, greek, grocery, mediterranean, restaura..."


## 12) Conclusion

In this experiment, evaluate the final ranking tables to identify which candidate
(Popularity, CF, CBF, or Weighted Hybrid) performs best on Top-K metrics.

If CF or CBF individually still beats the weighted hybrid, tune `HYBRID_ALPHA`
and/or add richer content features before retesting.

This notebook provides a complete, reproducible pipeline for building and evaluating
recommendation models on real-world sparse interaction data, with built-in support for
interpretability and serving-time non-repeat constraints.

